# Bankruptcy Prediction

**Classification Project 27 of 100** — Predict whether a company is at risk of bankruptcy.

EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.


## 2. Project Overview

Predict whether a company is at risk of bankruptcy using financial ratios and accounting-derived indicators. This is a classic credit risk / financial distress problem where extreme class imbalance is the norm.

The dataset contains dozens of financial ratio features. We use the Taiwan Economic Journal bankruptcy dataset.


## 3. Learning Objectives

1. Handling extreme class imbalance (bankruptcy is rare)
2. Working with high-dimensional financial ratio features
3. Understanding why ROC-AUC and recall matter more than accuracy
4. Feature selection on correlated financial indicators
5. LazyPredict and FLAML comparison on imbalanced data
6. Financial domain context for ML predictions


## 4. Problem Statement

> Given a company's financial ratios and accounting metrics, predict whether it will go bankrupt.

Binary classification. Recall and PR-AUC are critical — missing a bankruptcy is far costlier than a false alarm.


## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Banks / lenders | Avoid bad loans |
| Investors | Portfolio risk management |
| Regulators | Systemic risk monitoring |
| ML learners | Extreme imbalance + financial features |


## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | fedesoriano/company-bankruptcy-prediction |
| Rows | ~6,800 |
| Features | 95 financial ratios |
| Target | Bankrupt? (0/1) |
| Balance | Extremely imbalanced (~3% positive) |


## 7. Dataset Source and License Notes

- Taiwan Economic Journal, 1999-2009
- Kaggle: fedesoriano/company-bankruptcy-prediction
- License: CC0 Public Domain


## 8. Environment Setup


In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## 9. Imports


In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, ConfusionMatrixDisplay, RocCurveDisplay,
    classification_report, PrecisionRecallDisplay)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")


## 10. Configuration / Constants


In [ ]:
KAGGLE_SLUG = "fedesoriano/company-bankruptcy-prediction"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 120; MAX_ROWS = 50000
print(f"Kaggle slug: {KAGGLE_SLUG}")


## 11. Dataset Download or Loading


In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


## 12. Data Validation Checks


In [ ]:
print(f"Missing values total: {df_raw.isnull().sum().sum()}")
df = df_raw.copy()
target_candidates = [c for c in df.columns if any(h in c.lower() for h in ['bankrupt','bankruptcy','class','target'])]
TARGET = target_candidates[0] if target_candidates else df.columns[-1]
print(f"Target column: {TARGET}")
id_cols = [c for c in df.columns if c.lower() in ['id','index','unnamed: 0']]
if id_cols: df = df.drop(columns=id_cols)
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
print(f"Clean shape: {df.shape}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")


## 13. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue","coral"])
ax.set_title(f"Target Distribution: {TARGET}")
plt.tight_layout(); plt.show()


In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
if len(num_feats) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    for ax, col in zip(axes.flat, num_feats[:4]):
        for val in sorted(df[TARGET].unique()):
            subset = df[df[TARGET]==val][col].clip(lower=df[col].quantile(0.01), upper=df[col].quantile(0.99))
            ax.hist(subset.dropna(), bins=30, alpha=0.4, label=f"{TARGET}={val}")
        ax.set_title(col[:40]); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


In [ ]:
target_corr = df[num_feats + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print("Top 15 features by |correlation| with target:")
print(target_corr.head(15))


## 14. Target Analysis

Bankruptcy is extremely rare (~3%). Accuracy is misleading — we must focus on recall, F1, PR-AUC.


In [ ]:
print(df[TARGET].value_counts(normalize=True))
print(f"\nMajority baseline accuracy: {df[TARGET].value_counts(normalize=True).max():.1%}")
print(f"Positive class ratio: {df[TARGET].mean():.2%}")


## 15. Train / Validation / Test Split


In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), name=TARGET)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")


## 16. Preprocessing Strategy

All features are numeric (financial ratios). Impute with median and standardize.


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols)
], remainder="drop")
print(f"Numeric features: {len(num_cols)}")


## 17. Feature Engineering

With 95 pre-computed financial ratios, we remove near-zero-variance columns.


In [ ]:
print(f"Features before: {X_train.shape[1]}")
low_var = X_train.var(numeric_only=True)
low_var_cols = low_var[low_var < 1e-6].index.tolist()
if low_var_cols:
    print(f"Near-constant columns removed: {len(low_var_cols)}")
    X_train = X_train.drop(columns=low_var_cols)
    X_val = X_val.drop(columns=low_var_cols)
    X_test = X_test.drop(columns=low_var_cols)
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    preprocessor = ColumnTransformer(transformers=[
        ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols)
    ], remainder="drop")
print(f"Features after: {X_train.shape[1]}")


## 18. Baseline Model

Use class_weight='balanced' for LogReg and RF to handle extreme imbalance.


In [ ]:
results = {}
for name, clf in [("Dummy", DummyClassifier(strategy="most_frequent", random_state=SEED)),
                  ("LogReg", LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)),
                  ("RF", RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {"Accuracy": accuracy_score(y_val,yp), "F1": f1_score(y_val,yp,zero_division=0),
         "Recall": recall_score(y_val,yp,zero_division=0)}
    if hasattr(clf, "predict_proba"):
        try:
            yprob = pipe.predict_proba(X_val)[:,1]
            r["ROC-AUC"] = roc_auc_score(y_val, yprob)
            r["PR-AUC"] = average_precision_score(y_val, yprob)
        except: r["ROC-AUC"] = np.nan; r["PR-AUC"] = np.nan
    results[name] = r; print(f"{name}: {r}")


## 19. LazyPredict Benchmark


In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)


## 20. FLAML AutoML Run


In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task="classification", metric="f1",
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f"Best estimator: {automl.best_estimator}")
yp_fl = automl.predict(X_val_lp)
results["FLAML"] = {"Accuracy": accuracy_score(y_val, yp_fl),
                    "F1": f1_score(y_val, yp_fl, average='binary', zero_division=0)}
print(results["FLAML"])


## 21. Top 3 Model Selection

Gradient boosting with class imbalance handling.


In [ ]:
pos_ratio = y_train.sum() / len(y_train)
scale_weight = (1 - pos_ratio) / max(pos_ratio, 1e-6)
try:
    from lightgbm import LGBMClassifier; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBClassifier; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, is_unbalance=True, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=500, class_weight='balanced', random_state=SEED, n_jobs=-1)
if has_xgb:
    top3["XGBoost"] = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, scale_pos_weight=scale_weight, random_state=SEED, verbosity=0, n_jobs=-1, eval_metric="logloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3["RF_balanced"] = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', max_depth=10, random_state=SEED, n_jobs=-1)
print(f"Top 3: {list(top3.keys())}")


## 22. Final Training and Evaluation of Top 3


In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
labels = ["Healthy", "Bankrupt"]
final = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    ypr = mdl.predict_proba(X_te_t)[:,1]
    final[name] = {"Accuracy": accuracy_score(y_test,yp), "Precision": precision_score(y_test,yp,zero_division=0),
                   "Recall": recall_score(y_test,yp,zero_division=0), "F1": f1_score(y_test,yp,zero_division=0),
                   "ROC-AUC": roc_auc_score(y_test,ypr), "PR-AUC": average_precision_score(y_test,ypr)}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=labels))
pd.DataFrame(final).T


In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_te_t), ax=ax, cmap="Blues", display_labels=labels)
    ax.set_title(n)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m, X_te_t, y_test, ax=axes[0], name=n)
    PrecisionRecallDisplay.from_estimator(m, X_te_t, y_test, ax=axes[1], name=n)
axes[0].plot([0,1],[0,1],"k--",alpha=0.5); axes[0].set_title("ROC Curves")
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout(); plt.show()


## 23. Error Analysis


In [ ]:
bm = list(top3.values())[0]; ypb = bm.predict(X_te_t)
fn = (y_test.values==1) & (ypb==0); fp = (y_test.values==0) & (ypb==1)
print(f"False Negatives (missed bankruptcies): {fn.sum()}")
print(f"False Positives (false alarms): {fp.sum()}")
print(f"Total misclassified: {(y_test.values!=ypb).sum()}/{len(y_test)} ({(y_test.values!=ypb).mean():.1%})")


## 24. Interpretation and Business Insight

- **High recall is essential**: Missing a bankruptcy can mean millions in losses
- **Financial ratios are powerful**: Debt-to-equity and cash flow metrics are strong predictors
- **Class imbalance handling is critical**: Without it, models predict 'healthy' for everything
- **Threshold tuning matters**: Lower threshold to catch more at-risk companies


## 25. Limitations

1. Extreme class imbalance makes evaluation challenging
2. Historical data (1999-2009) may not generalize
3. No industry segmentation
4. Financial ratios are lagging indicators
5. Survivorship bias


## 26. How to Improve This Project

1. Add macroeconomic indicators
2. Industry-specific models
3. Time-aware validation
4. Anomaly detection complement
5. Ensemble with survival analysis


## 27. Production Considerations

- Quarterly batch scoring
- Regulatory compliance (explainability)
- Probability calibration for risk scoring
- Watchlist integration
- Model validation by risk teams


## 28. Common Mistakes

1. Using accuracy with 97% majority class
2. Not using class weights or oversampling
3. Ignoring PR-AUC
4. Leaking future financial data
5. Not considering FN vs FP cost asymmetry


## 29. Mini Challenge / Exercises

1. Apply SMOTE and compare with class_weight='balanced'
2. Tune threshold for recall > 80%
3. Use only top 10 features
4. Cost-sensitive model (FN=100x FP)
5. Try Isolation Forest as alternative


## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary — bankruptcy prediction |
| Dataset | ~6,800 companies, 95 financial ratios |
| Difficulty | Hard (extreme imbalance) |
| Key insight | Class imbalance handling is the #1 challenge |

Financial distress prediction with extreme imbalance, emphasis on recall/PR-AUC.
